In [24]:
import pandas as pd
import numpy as np
import json
import random as rnd
import string
import datetime
from dateutil.relativedelta import relativedelta
import os

Create Empty Data Frames for All CSVs We want.

In [25]:
last_trigger = {"Content" : datetime.date(2023, 6, 1), "Event": datetime.date(2023, 7, 1)}

In [26]:
#Function to return the probability of an event happening in a given day
def day_prob(pMin, pMax, dMin, dMax):
    #Simulate random day probabilities.
    return 1.0 - (1.0 - rnd.uniform(pMin, pMax)) ** (1.0 / rnd.randint(dMin, dMax))


def generate_random_deals(pMin,pMax, growth):
    #Simulate random partnership/referral deals.
    count = 0
    while rnd.uniform(0, 1) < (rnd.uniform(pMin,pMax) * growth):
        count += 1
    return count


def id_writer(type, number):

    with open("Rules_Tables/rules/tracker_id.json", "r", encoding="utf-8") as file:
        Tracker_ID = json.load(file)
    
    current_ID = int(Tracker_ID[type][-10:])
    new_ids = []

    for id in range(number):
        current_ID += 1
        new_ids.append(f"ID{current_ID:010d}")

    Tracker_ID[type] = f"ID{current_ID:010d}"

    with open("Rules_Tables/rules/tracker_id.json", "w", encoding="utf-8") as file:
        json.dump(Tracker_ID, file, indent = 4)
    
    return new_ids


In [ ]:
def content_creator(day):
    content_types = [
        "White_Paper",
        "Case_Study",
        "Blog_Post",
        "Guide",
        "Webinar",
        "Infographic"
        ]
    content_type = rnd.choice(content_types)
    campaign_id = id_writer("Campaign",1)
    campaign_name = f"Content_{day.dt.to_period("M")}_{content_type}"
    campaign_sub_type = content_type
    campaign_type = "Content" 

def event_creator(day):
    event_types = [
    "Trade_shows",
    "Conferences",
    "Networking_events",
    "Workshops",
    "Webinars",
    "Product_launches"
]
    

In [31]:
CONTENT_DECAY_DAYS = 5
EVENT_MIN_DAYS = 30
EVENT_SCALE = 120

def generate_daily_leads(today, growth, last_trigger):
    """Generate today's new leads by category."""
    leads = {
        "Organic": 0, "Content": 0, "Event": 0,
             "Partnership": 0, "Referral": 0}

    last_content_trigger = today - last_trigger["Content"]
    last_event_trigger = today - last_trigger["Event"]

    if today.weekday() < 5:  # Weekday
        leads["Organic"] = int(round(rnd.randint(0, 12) * growth, 0))

        if last_content_trigger.days <= CONTENT_DECAY_DAYS:
            leads["Content"] = int(round(
                (rnd.randint(10, 30) * growth) - (last_content_trigger.days), 0
            ))

        if rnd.uniform(0, 1) <= day_prob(0.25, 0.50, 30, 30):
            leads["Content"] = int(round(rnd.randint(10, 30) * growth, 0))
            last_trigger["Content"] = today

        if (
            rnd.uniform(0, 1)
            < ((last_event_trigger.days / EVENT_SCALE) * growth)
            and last_event_trigger.days > EVENT_MIN_DAYS
            ):
            leads["Event"] = int(round(rnd.randint(0, 200) * growth, 0))
            last_trigger["Event"] = today

        leads["Partnership"] = generate_random_deals(0.05, 0.1, growth)
        leads["Referral"] = generate_random_deals(0.05, 0.1, growth)

    else:  # Weekend
        leads["Organic"] = int(round(rnd.randint(0, 5) * growth, 0))
        if last_content_trigger.days <= CONTENT_DECAY_DAYS:
            leads["Content"] = int(round(rnd.randint(2, 5) * growth, 0))

    lead_list = []
    
    for type, value in leads.items():    
        #print(f"{type} = {value}")
        lead_list = lead_list + ([type] * value )
    
    lead_count = len(lead_list)
    rnd.shuffle(lead_list)

    new_leads = pd.DataFrame({
        "LeadID" : id_writer("Lead", lead_count),
        "Created Date": [today] * lead_count,
        "Lead Status": ['Marketing Qualified'] * lead_count,
        "Lead Source": lead_list,
        "Recent Lead Source": lead_list,
        "MQL Date":  [today] * lead_count
        })
    
    return new_leads


def daily_converstion(today, leads, last_trigger):

    lead_source_rules = pd.read_csv("Rules_Tables/rules/lead_source.csv")
    lead_source_rules['Randomizer'] = lead_source_rules.apply(lambda x: day_prob(x['Min'], x['Max'], 30, 100), axis = 1)

    last_content_trigger = today - last_trigger["Content"]

    working_leads = leads[leads["Lead Status"] != "Qualified"]

    if today.weekday() < 5:  # Weekday
        for index, lead in working_leads.iterrows(): 
            if lead['Lead Status'] == 'New' or lead['Lead Status'] == 'Marketing Qualified':
                if rnd.uniform(0,1)<.75:
                    leads.loc[index,'Lead Status'] = "Qualifying"
                    leads.loc[index,'Qualifying Date'] = today
                
                elif rnd.uniform(0,1)< rnd.uniform(.05,.2):
                    leads.loc[index,'Lead Status'] = "Disqualified"        

            elif lead['Lead Status'] == 'Qualifying':
                if rnd.uniform(0,1) < (lead_source_rules.loc[lead_source_rules['Name'] == lead['Lead Source'], 'Randomizer'].iloc[0]):
                    leads.loc[index,'Lead Status'] = "Qualified"
                    leads.loc[index,'Qualified Date'] = today

                elif ((today-lead['Qualifying Date']).days) > rnd.randint(60,90): 
                    leads.loc[index,'Lead Status'] = "Nurture"
                    leads.loc[index,'Nurture Date'] = today  
            
            elif lead['Lead Status'] == 'Nurture':
                if last_content_trigger.days <= CONTENT_DECAY_DAYS and rnd.uniform(0,1) < day_prob(.05,.1,90,270):
                    leads.loc[index,'Lead Status'] = 'Marketing Qualified'
                    leads.loc[index,'ReMQL Date'] = today
                    leads.loc[index,'Recent Lead Source'] = "Content"

In [28]:
START = datetime.date(2023, 7, 1)
today = datetime.date.today()
growth = 1.0
lead_df = pd.read_csv(
            "Rules_Tables/data/lead.csv",
            parse_dates=["Qualifying Date", "Qualified Date", "Nurture Date", "ReMQL Date"]
            )
delta = datetime.timedelta(days=1)

for i in range((today - START).days + 1):
    day = START + i * delta
    growth = 1 + ((day-START).days)/900
    new_lead_df = generate_daily_leads(day, growth, last_trigger)
    lead_df = pd.concat([lead_df, new_lead_df], ignore_index=True)
    daily_converstion(day, lead_df, last_trigger)

lead_df

,LeadID,Created Date,Lead Status,MQL Date,Qualifying Date,Nurture Date,Qualified Date,Lead Source,Lead Source Details,Rep ID,Rep Name,ReMQL Date,Recent Lead Source,Recent Lead Source Details,Lead ID
0,NaN,2023-07-01,Nurture,2023-07-01,2023-07-04,2023-09-11,NaN,Organic,NaN,NaN,NaN,NaN,Organic,NaN,ID0000000001
1,NaN,2023-07-01,Qualified,2023-07-01,2023-07-05,NaN,2023-08-04,Organic,NaN,NaN,NaN,NaN,Organic,NaN,ID0000000002
2,NaN,2023-07-01,Nurture,2023-07-01,2023-07-03,2023-09-11,NaN,Organic,NaN,NaN,NaN,NaN,Organic,NaN,ID0000000003
3,NaN,2023-07-02,Nurture,2023-07-02,2023-07-03,2023-09-12,NaN,Organic,NaN,NaN,NaN,NaN,Organic,NaN,ID0000000004
4,NaN,2023-07-03,Nurture,2023-07-03,2023-07-03,2023-09-07,NaN,Organic,NaN,NaN,NaN,NaN,Organic,NaN,ID0000000005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9561,NaN,2025-08-17,Qualifying,2025-08-17,2025-08-18,NaN,NaN,Organic,NaN,NaN,NaN,NaN,Organic,NaN,ID0000009562
9562,NaN,2025-08-17,Qualifying,2025-08-17,2025-08-18,NaN,NaN,Organic,NaN,NaN,NaN,NaN,Organic,NaN,ID0000009563
9563,NaN,2025-08-17,Qualifying,2025-08-17,2025-08-18,NaN,NaN,Organic,NaN,NaN,NaN,NaN,Organic,NaN,ID0000009564
9564,NaN,2025-08-18,Qualifying,2025-08-18,2025-08-18,NaN,NaN,Organic,NaN,NaN,NaN,NaN,Organic,NaN,ID0000009565


In [29]:
lead_df.dtypes

LeadID                        object
Created Date                  object
Lead Status                   object
MQL Date                      object
Qualifying Date               object
Nurture Date                  object
Qualified Date                object
Lead Source                   object
Lead Source Details           object
Rep ID                        object
Rep Name                      object
ReMQL Date                    object
Recent Lead Source            object
Recent Lead Source Details    object
Lead ID                       object
dtype: object

In [30]:
lead_df.to_csv("Rules_Tables/data/lead.csv", index=False)